# F02-P1 Select Area of Interest

Users upload a project-area polygon, which is analyzed using Rasterio. The vector polygon is rasterized against the `SEA_NBS_PATHWAY` reference grid, and the total project area is calculated by summing the area of pixels within the polygon.

Using the same raster grid and area-calculation method across subsequent analyses ensures consistency between project area, ecosystem area, disturbance, and benefit calculations, reducing discrepancies and potential user confusion.

In [ ]:
import geopandas as gpd
import numpy as np
import rasterio

from pyproj import Geod
from rasterio.mask import raster_geometry_mask

In [ ]:
def area_ha(mask, transform):

    total = 0.0

    for row in range(mask.shape[0]):

        n = mask[row].sum()

        if n == 0:
            continue

        north = transform.f + row * transform.e
        south = north + transform.e
        west = transform.c
        east = west + transform.a

        area, _ = GEOD.polygon_area_perimeter(
            [west, east, east, west],
            [north, north, south, south]
        )

        total += n * abs(area)

    return total / 10000


aoi = gpd.read_file(USER_AOI)
geom = aoi.geometry.union_all()

with rasterio.open(PATHWAY) as src:

    geom = (
        gpd.GeoSeries([geom], crs=aoi.crs)
        .to_crs(src.crs)
        .iloc[0]
    )

    outside, transform, _ = raster_geometry_mask(
        src,
        [geom.__geo_interface__],
        crop=True,
        all_touched=False
    )

project_area_ha = area_ha(
    ~outside,
    transform
)

print(f"Project area: {project_area_ha:,.2f} ha")